In [268]:
from langchain_core.output_parsers import JsonOutputParser
from src.llm_core.llm_core import llm
from src.llm_core.llm_prompt_base import LLMBase
from tqdm.auto import tqdm

# final_answer_prompt = """Ты — профессиональный генератор поисковых запросов для обучения ML моделей.
# Ты помогаешь обучать агента, который будет отвечать на вопросы пользователей у банкомата. 
# Сейчас ты генерируешь данные для обучения модели оркестратора - эта модель должна определять, на какую ручку отправить запрос пользователя.
# На вход ты получаешь изначальные запросы, которые уже относятся к какому-то агенту. Твоя задача - перефразировать их таким образом, чтобы они относились к целевому агенту. 

# # Задача:
# Твои входные данные:
# 1. Исходный вопрос пользователя: str // Исходный вопрос, который задал пользователь
# 2. Изначальный агент: str // Агент, который отвечает на исходный вопрос
# 3. Целевой агент: str // Агент, к которому вопрос должен быть сформирован

# # Требования к генерации:
# 0. Сгенерированный тобой вопрос должен сохранять общую стилистику исходного.
# 1. Ты должен сгенерировать такой вопрос, чтобы его могла выполнить именно **ЦЕЛЕВАЯ РУЧКА**. 
#     - Пример: 
#         - Изначальный запрос: хочу снять деньги со счета
#         - Изначальный агент: Агент выдачи (cashout_agent) - отвечает за снятие денежных средств в банкомате. Его можно попросить снять определенными купюрами (например, снять тысячу мелкими купюрами или снять тысячу сотками). 
#         - Целевой агент: Агент взноса (cashin_agent) - отвечает за внесение денежных средств в банкомате
#         - Ответ: хочу внести деньги на счет
# **ВАЖНО!!** Если в агенте написано, что он ТОЛЬКО делает действие, то вопрос должен быть таким, чтобы на него могло ответить именно ДЕЙСТВИЕ. К примеру, если написано, что агент должен вернуть ДЕЙСТВИЕ, ты не можешь в вопросах просить его оказать консультацию.
# 2. При этом ты обязан изменить так, чтобы объект вопроса не изменился (например, если речь идет о счете, то и сгенерированные тобой вопросы должны относиться **ИМЕННО** к счету. Если в оригинальном вопросе речь шла о балансе, то твой вопрос должен быть о балансе. и тд.)
# 2. Формат как для поисковой системы (без предложений-ответов)
# 3. Каждый вопрос должен уточнять разные аспекты
# 4. Избегай общих вопросов в стиле "расскажи всё о..."
# 6. Учти вопросы, которые уже были заданы! Твои вопросы не должны дублировать те, что уже были созданы! Если ты не можешь придумать новых вопросов, в поле questions верни пустой список.
# 7. Используй точные термины из контекста при генерации вопросов.
# 8. Если ты понимаешь, что запрос никак нельзя перефразировать, чтобы на него отвечал целевой агент, то верни пустой список.

# # Формат вывода:
# Ты должен вернуть JSON со следующими полями: 
# {{"reasoning": str, // Твои размышления, 
#   "questions": list[str], // Перефразированные вопросы, не более 3 штук!
# }}

# Сгенерируй вопросы:"""


final_answer_prompt = """# Роль
Ты — эксперт по генерации лингвистических данных для обучения ML-моделей. Твоя специфика — перефразирование пользовательских запросов в строго заданном контексте банкомата Сбера.

# Контекст задачи
Ты обучаешь модель-оркестратор, которая классифицирует запросы пользователей и направляет их соответствующему агенту-обработчику. Ты получаешь на вход три элемента:
1.  `исходный_запрос` (str) — исходная фраза пользователя.
2.  `исходный_агент` (str) — агент, который корректно обрабатывает исходный запрос.
3.  `целевой_агент` (str) — агент, для которого нужно адаптировать запрос.

Твоя задача — **строго переформулировать** `исходный_запрос` так, чтобы его смысловое ядро (интенция) было адаптировано под функционал `целевого_агента`, при этом сохранив стилистику и уровень детализации оригинала.

# Спецификации агентов
В своей работе ты должен неукоснительно руководствоваться следующими правилами для каждого агента:

-   **`cashin_agent` (Агент взноса)**
    -   **Функционал:** ТОЛЬКО физическое внесение наличных денег на счет через банкомат.
    -   **Запрещено:** Отвечать на вопросы о балансе, комиссиях, лимитах, "как внести?", а также выполнять любые другие действия, кроме приема купюр.

-   **`cashout_agent` (Агент выдачи)**
    -   **Функционал:** ТОЛЬКО физическая выдача наличных денег со счета, включая запрос на выдачу определенными купюрами (мелкими, крупными).
    -   **Запрещено:** Отвечать на вопросы о балансе, остатке, лимитах на снятие, "как снять?", а также выполнять любые другие действия.

-   **`balance_agent` (Агент баланса)**
    -   **Функционал:** ТОЛЬКО предоставление информации о текущем балансе на карте.
    -   **Запрещено:** Выполнять операции по снятию, внесению, переводу, а также сообщать историю операций или блокировать карту.

-   **`all_other_agent` (Агент прочих операций)**
    -   **Функционал:** Обработка всего, что не входит в функционал трех вышеуказанных агентов. Это включает: нерелевантные запросы (например, "погода"), неполные запросы (где не хватает данных для выполнения действия), запросы на отмену операции, консультации по продуктам.
    -   **Запрещено:** Отвечать на прямые запросы, относящиеся к взносу, снятию или проверке баланса.

# Критерии и правила генерации

1.  **Изменение объекта:** Основная цель запроса (интенция) должна быть изменена с исходного агента на целевого, но объект (например, "счет", "деньги", "карта") должен остаться тем же.
    -   *Пример:* `"хочу снять деньги со счета"` (cashout) -> `"хочу внести деньги на счет"` (cashin).

2.  **Стилистическое соответствие:** Сохраняй регистр, пунктуацию, уровень формальности и разговорный стиль исходного запроса. Если исходный запрос был с ошибками, можно сохранить аналогичный уровень грамотности.

3.  **Фокус на действии:** Вопрос должен формулироваться как прямое указание к действию, которое доступно целевому агенту, или как минимальный запрос, ведущий к этому действию. Избегай общих вопросов ("расскажи о...", "как это сделать?").

4.  **Запрет на дублирование:** Сгенерированные вопросы не должны повторять уже существующие данные в рамках текущей задачи. Если невозможно придумать уникальную вариацию, список должен быть пустым.

5.  **Проверка возможности преобразования:** Если функционал исходного и целевого агентов полностью несовместимы и преобразование невозможно без искажения смысла, которое привело бы к некорректной классификации, — верни пустой список.
    -   *Пример невозможного преобразования:* Запрос "Сколько у меня денег?" (balance_agent) нельзя осмысленно преобразовать для `cashin_agent`, так как тот не предоставляет информацию.

6.  **Количество:** Сгенерируй не более 3 (трех) вариантов перефразирования.

# Выходной формат
Ответ ТОЛЬКО в виде JSON без Markdown, с экранированными спецсимволами:
{{"negative_1": "string", "negative_2": "string", "negative_3": "string"}}
"""

class AnswerAgent(LLMBase):
    def make_user_prompt(self, user_question, init_agent, should_agent):
        message = [
            f"**Исходный вопрос пользователя:** {user_question}",
            f"**Изначальный агент:** {init_agent}",
            f"**Целевой агент** {should_agent}",
        ]
        user_prompt = "\n\n".join(message)
        messages = {"messages": [("user", user_prompt)]}
        return messages


answer_agent = AnswerAgent(llm, final_answer_prompt, JsonOutputParser())


In [269]:
import pandas as pd

In [270]:
data = pd.read_excel("state_messages_dataset.xlsx")
data = data.loc[data["category"] != "advisor_agent"].reset_index(drop=True)

In [271]:
data['category'].unique()

array(['all_other_agent', 'balance_agent', 'cashin_agent',
       'cashout_agent'], dtype=object)

In [272]:
categories = {"all_other_agent": "Агент обработки остальных операций - отвечает за нерелевантные запросы (которые не относятся к базовым операциям или консультациям по продуктам Сбера на банкомате), неполные (не хватает данных) запросы и отмены операций. Вопросы к этому агенту НЕ ДОЛЖНЫ касаться взноса денег, снятия денег или подробностей по счету клиента.", 
              "cashin_agent": "Агент взноса - отвечает за внесение денежных средств в банкомате. ТОЛЬКО вносит деньги на счет, он НЕ ОТВЕЧАЕТ на вопросы о балансе, возможности внесения денег и тд. Ему НЕЛЬЗЯ задавать вопросы формата 'как', он на них НЕ ОТВЕЧАЕТ.", 
              "cashout_agent": "Агент выдачи - отвечает за снятие денежных средств в банкомате. Его можно попросить снять определенными купюрами (например, снять тысячу мелкими купюрами или снять тысячу сотками). ТОЛЬКО снимает деньги, он НЕ ОТВЕЧАЕТ на вопросы о балансе, возможности снятия и тд. Ему НЕЛЬЗЯ задавать вопросы формата 'как', он на них НЕ ОТВЕЧАЕТ.",
              "balance_agent": "Агент баланса (balance_agent) - отвечает за показ баланса на карте клиента. ТОЛЬКО показывает баланс. Он не может ничего сделать с деньгами кроме как показать баланс. (Он НЕ МОЖЕТ снять или внести деньги)"
             }

In [306]:
# num = 100
num = data.loc[data["category"] == "cashin_agent"].index[0]
query = data.loc[num, "text"]
init_agent = categories[data.loc[num, "category"]]
print(data.loc[num, "category"])
should_agent = categories["cashout_agent"]

cashin_agent


In [286]:
answer_agent.invoke(user_question=query, init_agent=init_agent, should_agent=should_agent)

{'negative_1': 'отсканируй qr и покажи меню',
 'negative_2': 'отсканируй qr и отмени операцию',
 'negative_3': 'отсканируй qr и покажи акции'}

In [287]:
query

'отсканируй qr и прими наличные'

In [308]:
# answer_agent.invoke(user_question="бери бабосики", init_agent=init_agent, should_agent=should_agent)

In [325]:
data_test_all_other_agent = data.loc[data["category"] == "all_other_agent"]
data_test_cashin_agent = data.loc[data["category"] == "cashin_agent"]
data_test_cashout_agent = data.loc[data["category"] == "cashout_agent"]
data_test_balance_agent = data.loc[data["category"] == "balance_agent"]

In [326]:
data_test_for_balance = pd.concat([data_test_all_other_agent, data_test_cashin_agent, data_test_cashout_agent], ignore_index=True)
data_test_for_cashin = pd.concat([data_test_all_other_agent, data_test_cashout_agent, data_test_balance_agent], ignore_index=True)
data_test_for_cashout = pd.concat([data_test_all_other_agent, data_test_cashin_agent, data_test_balance_agent], ignore_index=True)
data_test_for_other = pd.concat([data_test_cashin_agent, data_test_cashout_agent, data_test_balance_agent], ignore_index=True)

In [327]:
# Замена на баланс
all_messages = []
should_agent = categories["balance_agent"]
for query, cat in tqdm(zip(data_test_for_balance["text"].to_list(), data_test_for_balance["category"].to_list())):
    init_category = categories[cat]
    all_messages.append(answer_agent.make_user_prompt(user_question=query, init_agent=init_category, should_agent=should_agent))
res = await answer_agent.batch(all_messages, concurrency=100)
data_test_for_balance["negatives"] = res
data_test_for_balance["target_category"] = "balance_agent" 

0it [00:00, ?it/s]

Exception ignored in: <function tqdm.__del__ at 0x0000017D062A02C0>
Traceback (most recent call last):
  File "D:\Anaconda\envs\sledopyt\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "D:\Anaconda\envs\sledopyt\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm' object has no attribute 'disp'


In [328]:
# Замена на взнос
all_messages = []
should_agent = categories["cashin_agent"]
for query, cat in tqdm(zip(data_test_for_cashin["text"].to_list(), data_test_for_cashin["category"].to_list())):
    init_category = categories[cat]
    all_messages.append(answer_agent.make_user_prompt(user_question=query, init_agent=init_category, should_agent=should_agent))
res = await answer_agent.batch(all_messages, concurrency=100)
data_test_for_cashin["negatives"] = res
data_test_for_cashin["target_category"] = "cashin_agent" 

0it [00:00, ?it/s]

In [329]:
# Замена на снятие
all_messages = []
should_agent = categories["cashout_agent"]
for query, cat in tqdm(zip(data_test_for_cashout["text"].to_list(), data_test_for_cashout["category"].to_list())):
    init_category = categories[cat]
    all_messages.append(answer_agent.make_user_prompt(user_question=query, init_agent=init_category, should_agent=should_agent))
res = await answer_agent.batch(all_messages, concurrency=100)
data_test_for_cashout["negatives"] = res
data_test_for_cashout["target_category"] = "cashout_agent" 

0it [00:00, ?it/s]

In [330]:
# Замена на мусор
all_messages = []
should_agent = categories["all_other_agent"]
for query, cat in tqdm(zip(data_test_for_other["text"].to_list(), data_test_for_cashin["category"].to_list())):
    init_category = categories[cat]
    all_messages.append(answer_agent.make_user_prompt(user_question=query, init_agent=init_category, should_agent=should_agent))
res = await answer_agent.batch(all_messages, concurrency=100)
data_test_for_other["negatives"] = res
data_test_for_other["target_category"] = "all_other_agent" 

0it [00:00, ?it/s]

In [331]:
data_test = pd.concat([data_test_for_balance, data_test_for_cashin, data_test_for_cashout, data_test_for_other], ignore_index=True)

In [332]:
data_test.to_excel("data_negs.xlsx")

# Плохие паттерны

In [354]:
from langchain_core.output_parsers import JsonOutputParser
from src.llm_core.llm_core import llm
from src.llm_core.llm_prompt_base import LLMBase
from tqdm.auto import tqdm

# final_answer_prompt = """Ты — профессиональный генератор поисковых запросов для обучения ML моделей.
# Ты помогаешь обучать агента, который будет отвечать на вопросы пользователей у банкомата. 
# Сейчас ты генерируешь данные для обучения модели оркестратора - эта модель должна определять, на какую ручку отправить запрос пользователя.
# На вход ты получаешь изначальные запросы, которые уже относятся к какому-то агенту. Твоя задача - перефразировать их таким образом, чтобы они относились к целевому агенту. 

# # Задача:
# Твои входные данные:
# 1. Исходный вопрос пользователя: str // Исходный вопрос, который задал пользователь
# 2. Изначальный агент: str // Агент, который отвечает на исходный вопрос
# 3. Целевой агент: str // Агент, к которому вопрос должен быть сформирован

# # Требования к генерации:
# 0. Сгенерированный тобой вопрос должен сохранять общую стилистику исходного.
# 1. Ты должен сгенерировать такой вопрос, чтобы его могла выполнить именно **ЦЕЛЕВАЯ РУЧКА**. 
#     - Пример: 
#         - Изначальный запрос: хочу снять деньги со счета
#         - Изначальный агент: Агент выдачи (cashout_agent) - отвечает за снятие денежных средств в банкомате. Его можно попросить снять определенными купюрами (например, снять тысячу мелкими купюрами или снять тысячу сотками). 
#         - Целевой агент: Агент взноса (cashin_agent) - отвечает за внесение денежных средств в банкомате
#         - Ответ: хочу внести деньги на счет
# **ВАЖНО!!** Если в агенте написано, что он ТОЛЬКО делает действие, то вопрос должен быть таким, чтобы на него могло ответить именно ДЕЙСТВИЕ. К примеру, если написано, что агент должен вернуть ДЕЙСТВИЕ, ты не можешь в вопросах просить его оказать консультацию.
# 2. При этом ты обязан изменить так, чтобы объект вопроса не изменился (например, если речь идет о счете, то и сгенерированные тобой вопросы должны относиться **ИМЕННО** к счету. Если в оригинальном вопросе речь шла о балансе, то твой вопрос должен быть о балансе. и тд.)
# 2. Формат как для поисковой системы (без предложений-ответов)
# 3. Каждый вопрос должен уточнять разные аспекты
# 4. Избегай общих вопросов в стиле "расскажи всё о..."
# 6. Учти вопросы, которые уже были заданы! Твои вопросы не должны дублировать те, что уже были созданы! Если ты не можешь придумать новых вопросов, в поле questions верни пустой список.
# 7. Используй точные термины из контекста при генерации вопросов.
# 8. Если ты понимаешь, что запрос никак нельзя перефразировать, чтобы на него отвечал целевой агент, то верни пустой список.

# # Формат вывода:
# Ты должен вернуть JSON со следующими полями: 
# {{"reasoning": str, // Твои размышления, 
#   "questions": list[str], // Перефразированные вопросы, не более 3 штук!
# }}

# Сгенерируй вопросы:"""


final_answer_prompt = """# Роль
Ты — эксперт по генерации лингвистических данных для обучения ML-моделей. 
Твоя специфика — перефразирование пользовательских запросов в строго заданном контексте банкомата Сбера.

# Контекст задачи
Ты обучаешь модель-оркестратор, которая классифицирует запросы пользователей и направляет их соответствующему агенту-обработчику. Ты получаешь на вход три элемента:
1.  `исходный_запрос` (str) — исходная фраза пользователя.
2.  `исходный_агент` (str) — агент, который корректно обрабатывает исходный запрос.
3.  `тематика` (str) — запретная тематика, которую агент должен фильтровать.

Твоя задача — **строго переформулировать** `исходный_запрос` так, чтобы его смысловое ядро (интенция) осталось неизменным, но запрос был бы испорчен запрещенными словами.

# Спецификации агентов
В своей работе ты должен неукоснительно руководствоваться следующими правилами для каждого агента:

-   **`cashin_agent` (Агент взноса)**
    -   **Функционал:** ТОЛЬКО физическое внесение наличных денег на счет через банкомат.
    -   **Запрещено:** Отвечать на вопросы о балансе, комиссиях, лимитах, "как внести?", а также выполнять любые другие действия, кроме приема купюр.

-   **`cashout_agent` (Агент выдачи)**
    -   **Функционал:** ТОЛЬКО физическая выдача наличных денег со счета, включая запрос на выдачу определенными купюрами (мелкими, крупными).
    -   **Запрещено:** Отвечать на вопросы о балансе, остатке, лимитах на снятие, "как снять?", а также выполнять любые другие действия.

-   **`balance_agent` (Агент баланса)**
    -   **Функционал:** ТОЛЬКО предоставление информации о текущем балансе на карте.
    -   **Запрещено:** Выполнять операции по снятию, внесению, переводу, а также сообщать историю операций или блокировать карту.

-   **`all_other_agent` (Агент прочих операций)**
    -   **Функционал:** Обработка всего, что не входит в функционал трех вышеуказанных агентов. Это включает: нерелевантные запросы (например, "погода"), неполные запросы (где не хватает данных для выполнения действия), запросы на отмену операции, консультации по продуктам.
    -   **Запрещено:** Отвечать на прямые запросы, относящиеся к взносу, снятию или проверке баланса.

# Критерии и правила генерации

1.  **Изменение объекта:** Основная цель запроса (интенция) должна быть неизменная, но при этом запрос должен быть испорчен словами из запретной тематики. 
    -   *Пример:* 
        - исходный_запрос `"хочу снять деньги со счета"` (cashout) 
        - тематика: секс 
        -> `"хочу снять деньги со счета и как следует оттянуться с блядями"`
    -   *Пример:* 
        - исходный_запрос `"хочу снять деньги со счета"` (cashout) 
        - тематика: знаменитости и политики 
        -> `"хочу снять деньги со счета путина"`

2.  **Стилистическое соответствие:** Сохраняй регистр, пунктуацию, уровень формальности и разговорный стиль исходного запроса. Если исходный запрос был с ошибками, можно сохранить аналогичный уровень грамотности.

3.  **Фокус на действии:** Вопрос должен формулироваться как прямое указание к действию, которое доступно целевому агенту, или как минимальный запрос, ведущий к этому действию. Избегай общих вопросов ("расскажи о...", "как это сделать?").

4.  **Запрет на дублирование:** Сгенерированные вопросы не должны повторять уже существующие данные в рамках текущей задачи. Если невозможно придумать уникальную вариацию, список должен быть пустым.

5.  **Запретные фразы:** используй разнообразные и разговорные запретные фразы. Не стесняйся графичности, эти фразы нужны для обучения модели цензора.

6.  **Количество:** Сгенерируй не более 3 (трех) вариантов перефразирования. Генерируй разнообразные фразы.

# Выходной формат
Ответ ТОЛЬКО в виде JSON без Markdown, с экранированными спецсимволами:
{{"negative_1":
        {{"query": "string", 
          "bad_word": "string"
        }}, 
  "negative_2": 
        {{"query": "string", 
          "bad_word": "string"
        }},  
  "negative_3": 
        {{"query": "string", 
          "bad_word": "string"
        }}, 
        }}
"""

class BadAgent(LLMBase):
    def make_user_prompt(self, user_question, init_agent, topic):
        message = [
            f"**Исходный вопрос пользователя:** {user_question}",
            f"**Изначальный агент:** {init_agent}",
            f"**Тематика** {topic}",
        ]
        user_prompt = "\n\n".join(message)
        messages = {"messages": [("user", user_prompt)]}
        return messages


bad_words_agent = BadAgent(llm, final_answer_prompt, JsonOutputParser())


In [374]:
topics = ["алкоголь", "наркотики", "В.В. Путин", "Г.О. Греф", "Илон Маск", 
          "политика", "терроризм", "оружие", "межрасовые распри", 
          "секс", "порно", "педофилия", "инцест", "нацизм", 'коммунизм',
            'социализм', 'капитализм', 'либерализм', 'консерватизм', 'национализм',
            'фашизм', 'расизм', 'экстремизм']

In [375]:
data_test_all_other_agent = data.loc[data["category"] == "all_other_agent"].sample(10, replace=False, random_state=42)
data_test_cashin_agent = data.loc[data["category"] == "cashin_agent"].sample(10, replace=False, random_state=42)
data_test_cashout_agent = data.loc[data["category"] == "cashout_agent"].sample(10, replace=False, random_state=42)
data_test_balance_agent = data.loc[data["category"] == "balance_agent"].sample(10, replace=False, random_state=42)

In [376]:
data_test = pd.concat([data_test_all_other_agent, data_test_cashin_agent, data_test_cashout_agent, data_test_balance_agent], ignore_index=True)

In [377]:
all_messages = []
all_messages_df = []
for query, cat in tqdm(zip(data_test["text"].to_list(), data_test["category"].to_list())):
    category = categories[cat]
    for topic in tqdm(topics):
        all_messages.append(bad_words_agent.make_user_prompt(user_question=query, init_agent=category, topic=topic))
        all_messages_df.append([query, cat, topic])
res = await bad_words_agent.batch(all_messages, concurrency=100)
all_messages_df = pd.DataFrame(all_messages_df, columns=["text", "category", "topic"])
all_messages_df["bad_words"] = res
all_messages_df.to_excel("bad_words.xlsx")

0it [00:00, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

...

  0%|          | 0/23 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

In [386]:


final_answer_prompt = """# Роль
Ты — эксперт по генерации лингвистических данных для обучения ML-моделей. 
Твоя специфика — создание данных для обучения модели цензора

# Контекст задачи
Ты обучаешь модель-цензор, которая должна 
3.  `тематика` (str) — запретная тематика, которую агент должен фильтровать.

Твоя задача — сгенерировать до 50 слов, которые модель цензор должна детектить и засекать в зависимости от данной тебе тематики.

# Критерии и правила генерации

1.  **Запрет на дублирование:** Сгенерированные вопросы не должны повторять уже существующие данные в рамках текущей задачи. Если невозможно придумать уникальную вариацию, список должен быть пустым.
2.  **Запретные фразы:** используй разнообразные и разговорные запретные фразы. Не стесняйся графичности, эти фразы нужны для обучения модели цензора.
3.  **Количество:** Сгенерируй не менее 10 вариантов. Генерируй разнообразные фразы.

# Выходной формат
Ответ ТОЛЬКО в виде JSON без Markdown, с экранированными спецсимволами:
{{"stop_words": list[str]}}
"""

class BadAgent(LLMBase):
    def make_user_prompt(self, topic):
        message = [
            f"**Тематика** {topic}",
        ]
        user_prompt = "\n\n".join(message)
        messages = {"messages": [("user", user_prompt)]}
        return messages


bad_words_agent = BadAgent(llm, final_answer_prompt, JsonOutputParser())

In [387]:
topics = ["алкоголь", "наркотики", "В.В. Путин", "Г.О. Греф", "Илон Маск", 
          "политика", "терроризм", "оружие", "межрасовые распри", 
          "секс", "порно", "педофилия", "инцест", "нацизм", 'коммунизм',
            'социализм', 'капитализм', 'либерализм', 'консерватизм', 'национализм',
            'фашизм', 'расизм', 'экстремизм']

In [388]:
bad_words_agent.invoke(topic='нацизм')

{'stop_words': ['хайль гитлер',
  'зиг хайль',
  'третий рейх',
  'арийская раса',
  'унтерменш',
  'окончательное решение',
  'хрустальная ночь',
  'свастика',
  'фашистская символика',
  'белое превосходство',
  'расовое превосходство',
  'антисемитская пропаганда',
  'национал-социализм',
  'фюрер',
  'рейхсфюрер',
  'гестапо',
  'СС',
  'СА',
  'нацистское приветствие',
  'расово неполноценный',
  'еврейский заговор',
  'протоколы сионских мудрецов',
  'расовое смешение',
  'расово чистый',
  'нацистская идеология',
  'гитлерюгенд',
  'концлагерь',
  'холокост отрицание',
  'неонацист',
  'нацистские лозунги',
  'расовые законы',
  'нордическая раса',
  'антиславянская риторика',
  'расовый враг',
  'нацистская пропаганда',
  'расово неполноценные народы',
  'нацистские символы',
  'расовое преследование',
  'нацистские марши',
  'расовые теории',
  'нацистская Германия',
  'расовое законодательство',
  'нацистские преступления',
  'расовые чистки',
  'нацистская символика',
  'рас

In [389]:
bad_words_agent.invoke(topic='секс')

{'stop_words': ['белая раса превыше всего',
  'евреи правят миром',
  'газовая камера для неполноценных',
  'хейль Гитлер',
  'окончательное решение еврейского вопроса',
  'арийская чистота крови',
  'унтерменши должны быть уничтожены',
  'слава третьему рейху',
  'жидомасонский заговор',
  'цыганская нечисть',
  'зиг хайль',
  'расовое превосходство арийцев',
  'лагеря смерти для унтерменшей',
  'еврейская чума',
  'чистка расы от дегенератов',
  'славяне - рабы',
  'холокост - выдумка',
  'нацистское приветствие',
  'евгеника для улучшения расы',
  'протоколы сионских мудрецов',
  'расовая гигиена',
  'антисемитизм как долг',
  'недочеловеки не имеют прав',
  'арийская империя',
  'кровь и почва',
  'национал-социалистическая революция',
  'еврейский большевизм',
  'расовый враг',
  'чистка нации от примесей',
  'смерть евреям']}

# Синонимы

In [191]:
from langchain_core.output_parsers import JsonOutputParser
from src.llm_core.llm_core import llm
from src.llm_core.llm_prompt_base import LLMBase

prompt_syn_1 ='''
Ты — эксперт по генерации тренировочных датасетов для embedding-моделей, специализирующихся на задаче sentence similarity. Датасеты имеют формат (anchor, positive) pairs.

**Входные данные:**
- anchor: вопрос пользователя
- Изначальный агент: агент, который должен ответить на этот вопрос. 

**Задача:**
Сгенерируй 2 positive вопроса-синонима к anchor, которые:
1. Спрашивают ту же информацию, что и anchor, но другими словами
    - Если в изначальном вопросе спрашивается КАК что-то сделать, то и измененный запрос тоже должен спрашивать, как что-то сделать
    - Если изначальный вопрос просит что-то СДЕЛАТЬ, то и измененный вопрос должен просить это. Если просит что-то СДЕЛАТЬ, тебе запрещено спрашивать КАК что-то сделать
    - На измененный вопрос должен ответить ТОТ ЖЕ агент, что для anchor вопроса.
2. Написаны естественным пользовательским языком. Стилистика должна 100% соответсовать изначальному запросу. 
    - Если в изначальном запросе используется много просторечной лексики или мата - используй их тоже. Твои запросы должна максимально быть похожими по стилистике на естественные запросы пользователей.
3. Допускают упрощения текста, по типу:
    - Использование аббривеатур (например 'Программа долгосрочных сбережений' → 'ПДС')
    - Сокращение (например 'кредитная карта' → 'кредитка')
    - Упущение стоп-слов (какой, как итд.), не несущих смысловой нагрузки (например 'какой процент по карте' → 'процент по карте')
4. Если увидишь в запросе *** - так обозначают матерные слова (бля, хуй, ебать и тд).

**Формат ответа:**
Ответ ТОЛЬКО в виде JSON без Markdown, с экранированными спецсимволами:
{{"positive_1": "string", "positive_2": "string", "positive_3": "string"}}
'''


class SynonymAgent(LLMBase):
    def make_user_prompt(self, user_question, init_agent):
        message = [
            f"**anchor:** {user_question}",
            f"**Изначальный агент:** {init_agent}",
        ]
        user_prompt = "\n\n".join(message)
        messages = {"messages": [("user", user_prompt)]}
        return messages


synonym_agent = SynonymAgent(llm, prompt_syn_1, JsonOutputParser())


In [192]:
# synonym_agent.invoke(user_question=query)

In [193]:
data_test_all_other_agent = data.loc[data["category"] == "all_other_agent"].sample(10, replace=False, random_state=42)
data_test_cashin_agent = data.loc[data["category"] == "cashin_agent"].sample(10, replace=False, random_state=42)
data_test_cashout_agent = data.loc[data["category"] == "cashout_agent"].sample(10, replace=False, random_state=42)
data_test_balance_agent = data.loc[data["category"] == "balance_agent"].sample(10, replace=False, random_state=42)

In [194]:
data_test = pd.concat([data_test_all_other_agent, data_test_cashin_agent, data_test_cashout_agent, data_test_balance_agent], ignore_index=True)

In [203]:
all_messages = []
for query, cat in tqdm(zip(data_test["text"].to_list(), data_test["category"].to_list())):
    category = categories[cat]
    all_messages.append(synonym_agent.make_user_prompt(user_question=query, init_agent=category))

0it [00:00, ?it/s]

In [204]:
res = await synonym_agent.batch(all_messages, concurrency=100)

In [205]:
data_test["test_res"] = res

In [206]:
data_test.to_excel("synonyms_test.xlsx")

In [208]:
# data_test

In [209]:
all_messages = []
for query, cat in tqdm(zip(data["text"].to_list(), data["category"].to_list())):
    category = categories[cat]
    all_messages.append(synonym_agent.make_user_prompt(user_question=query, init_agent=category))

0it [00:00, ?it/s]

In [212]:
res = await synonym_agent.batch(all_messages, concurrency=100)

In [214]:
data["synonyms"] = res

In [215]:
data.to_excel("synonyms.xlsx")

In [216]:
data

,text,category,synonyms
0,привет,all_other_agent,"{'positive_1': 'здарова', 'positive_2': 'салют..."
1,5 денег значить сбережения. И вот наличие Сбер...,all_other_agent,{'positive_1': '5 денег значит накопления. А С...
2,Ты не устанешь от моих вопросов?,all_other_agent,"{'positive_1': 'Тебе не надоест, что я постоян..."
3,что такое сбер притча,all_other_agent,"{'positive_1': 'сбер притча это что', 'positiv..."
4,Посоветуй духи на лето.,all_other_agent,"{'positive_1': 'Какие духи выбрать для лета?',..."
...,...,...,...
1289,Буду снимать 700,cashout_agent,"{'positive_1': 'Сними 700', 'positive_2': 'Сня..."
1290,Желаю снять проверить денег,cashout_agent,"{'positive_1': 'Хочу снять денег проверить', '..."
1291,Снять зарплату,cashout_agent,"{'positive_1': 'Снять бабки', 'positive_2': 'С..."
1292,Все деньгги на счету снять,cashout_agent,"{'positive_1': 'Снять всю наличку со счёта', '..."
